In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from nanodrz import data
import torchaudio
import torch
from nanodrz.utils import play, resample, visualise_annotation

In [ ]:
audio, sr = torchaudio.load("/home/harry/test.wav")
audio = audio.sum(dim=0)[None]
audio = audio / audio.abs().max()
audio = resample(sr, 16000, audio)
audio = audio[:,:16000*25]

In [ ]:
from denoiser import pretrained

denoiser =  pretrained.dns64().cpu().eval()

@torch.inference_mode()
def denoise(audio_file, sr=None):
    global denoiser
    if type(audio_file) is str: 
        audio, sr = torchaudio.load(audio_file)
    else:
        audio = audio_file
        assert sr is not None, "You must provide sample rate for loaded audio"
    
    audio = audio.sum(dim=0, keepdim=True)
    # audio = resample(sr, denoiser.sample_rate, audio)
    B = 40
    denoiser = denoiser.cuda()
    print(denoiser.sample_rate)
    wav = audio.split(B*denoiser.sample_rate, dim=1)
    denoised = []
    for w in wav:
        denoised.append(denoiser(w.cuda()))
    denoiser = denoiser.cpu()
    denoised = torch.cat(denoised, dim=-1)
    return denoised

denoised = denoise(audio, sr)

In [ ]:
from nanodrz.model import DiarizeGPT, Config

device = torch.device("cuda:1")
torch.cuda.set_device(device)
ckpt = torch.load("/home/harry/runs/nanodrz/1706953646/0030000.pt", map_location=device)
config = Config(**ckpt["config"])
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
nlabels = model.generate(audio[:,:20*16000].cuda(), temperature=1, max_steps=3*12)
visualise_annotation(nlabels)
play(audio[:,:20*16000])